In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score,
                              ConfusionMatrixDisplay, precision_recall_curve)
import matplotlib.pyplot as plt

X_train_smote = joblib.load('../models/X_train_smote.pkl')
y_train_smote = joblib.load('../models/y_train_smote.pkl')
X_test = joblib.load('../models/X_test.pkl')
y_test = joblib.load('../models/y_test.pkl')

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1  # uses all CPU cores, speeds up training
)
rf_model.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print("PR-AUC:", average_precision_score(y_test, y_proba_rf))

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))
print("PR-AUC:", average_precision_score(y_test, y_proba_xgb))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = pd.DataFrame({
    'Model': ['Logistic Regression (SMOTE)', 'Random Forest', 'XGBoost'],
    'Precision': [
        precision_score(y_test, joblib.load('../models/smote_logreg.pkl').predict(X_test)),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    'Recall': [
        recall_score(y_test, joblib.load('../models/smote_logreg.pkl').predict(X_test)),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    'F1': [
        f1_score(y_test, joblib.load('../models/smote_logreg.pkl').predict(X_test)),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, joblib.load('../models/smote_logreg.pkl').predict_proba(X_test)[:,1]),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_xgb)
    ]
})
results

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X_train_smote.columns)
importances.sort_values(ascending=False).head(10).plot(kind='barh')
plt.title('Top 10 Features - XGBoost')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
joblib.dump(xgb_model, '../models/final_model.pkl')